# Solution: Temperature Seasonality Decomposition

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error


In [ ]:
import matplotlib as mpl

UB = {"Brand Blue": "#175CFF", "Medium Blue": "#7BA2FF"}
UN = {"Black": "#0A0B0F", "Dark Gray": "#5A5C62"}
mpl.rcParams["lines.linewidth"] = 3
mpl.rcParams["axes.linewidth"] = 2


In [ ]:
DATA_PATH = "../data/melbourne_temps.csv"


In [ ]:
def load_temps(path):
    df = pd.read_csv(path, parse_dates=["Date"])
    df = df.set_index("Date")
    return df["Temp"]

temps = load_temps(DATA_PATH)
print(f"Rows: {len(temps)}")
temps.head()


In [ ]:
def extract_seasonality(series):
    return series.groupby(series.index.dayofyear).mean()

seasonal_profile = extract_seasonality(temps)
print(f"Profile length: {len(seasonal_profile)}")
seasonal_profile.head()


In [ ]:
def seasonal_baseline_forecast(train, seasonal_profile, horizon):
    future_days = pd.date_range(train.index.max() + pd.Timedelta(days=1), periods=horizon, freq="D")
    preds = [seasonal_profile[d.timetuple().tm_yday] for d in future_days]
    return pd.Series(preds, index=future_days)

train = temps.iloc[:-365]
test = temps.iloc[-365:]
forecast = seasonal_baseline_forecast(train, seasonal_profile, 365)
print(f"MAE: {mean_absolute_error(test, forecast):.2f} C")


In [ ]:
def plot_seasonal_forecast(train, test, forecast):
    fig, ax = plt.subplots(figsize=(14, 6))
    ax.plot(train.index, train.values, color=UN["Black"], label="Train")
    ax.plot(test.index, test.values, color=UB["Brand Blue"], label="Actual")
    ax.plot(forecast.index, forecast.values, color=US["Orange"], linestyle="--", label="Seasonal Baseline")
    ax.set_title("Melbourne Temperatures: Seasonal Baseline Forecast", fontsize=18, fontweight="bold")
    ax.set_ylabel("Temperature (C)", fontsize=16)
    ax.tick_params(axis="both", labelsize=14)
    ax.legend()
    plt.tight_layout()
    plt.show()

plot_seasonal_forecast(train, test, forecast)
